# Noise floor from the 2022 and 2023 epoch pair

The two acquisitions are one year apart at comparable return density. One year
of true canopy growth is small against the expected measurement error, so the
spread of their difference over stable cells estimates that error directly.
The result gates whether span-scale change is detectable at all.

Point clouds are read by bounding box over HTTPS. No point-cloud file is
downloaded.

PDAL is built into its own prefix rather than into the Colab base
environment, which is pinned to the runtime's own Python version.

In [ ]:
TILES = 4
BRANCH = "feat/lidar-truth-pipeline"
REPO = "https://github.com/Lakshaycodes08/CanopyGuard-AI.git"
WORK = "/content/CanopyGuard-AI"

import os

os.environ["TILES"] = str(TILES)
os.environ["BRANCH"] = BRANCH
os.environ["REPO"] = REPO

## Fetch the code

Only the bootstrap script is needed here. It clones or updates the repository
itself, provisions the environment, and runs the measurement.

In [ ]:
!curl -sSfL "https://raw.githubusercontent.com/Lakshaycodes08/CanopyGuard-AI/$BRANCH/scripts/colab_bootstrap.py" -o /content/bootstrap.py
!head -3 /content/bootstrap.py

## Run

Start with a small `TILES` to prove the path, then raise it. A tile whose
surfaces already exist is skipped, so a later run continues rather than
restarting.

The per-tile point counts print as the build proceeds. The table and the gate
print at the end.

In [ ]:
!python /content/bootstrap.py

## Result

The gate decides whether span-scale change is detectable. Do not adjust the
thresholds after seeing the numbers; they are in `configs/lidar.yaml`.

In [ ]:
import json
from pathlib import Path

report = Path("/content/truth/noise_floor.json")
if report.exists():
    result = json.loads(report.read_text())
    print("scale_m    sigma   lod95    mean        cells")
    for row in result["rows"]:
        print(f"{row['scale_m']:>7.0f} {row['sigma_m']:>8.3f} {row['lod95_m']:>7.3f} "
              f"{row['mean_m']:>+7.3f} {row['cells']:>12,.0f}")
    print(f"\ndecay exponent {result['rows'][0]['decay_exponent']:.3f}")
    print(f"tiles accepted {result['gate']['accepted_tiles']} of {result['gate']['evaluated_tiles']}")
    for name, passed in result["gate"]["checks"].items():
        print(f"{'PASS' if passed else 'FAIL'}  {name}")
    print(f"\nGATE {'PASS' if result['gate']['passed'] else 'FAIL'}")
else:
    print("No report yet. The run above did not reach the measurement step.")

## Keep the outputs

Drive is optional. The JSON is small enough to copy out of the cell above.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/CanopyGuard/truth"
!cp /content/truth/*.json "/content/drive/MyDrive/CanopyGuard/truth/" 2>/dev/null || true
!cp /content/truth/chm_*.tif "/content/drive/MyDrive/CanopyGuard/truth/" 2>/dev/null || true
!du -sh "/content/drive/MyDrive/CanopyGuard/truth" 2>/dev/null || echo "nothing copied yet"